# S23 · Brazo robótico: simulador y ventosa (DOBOT Magician)

**Objetivo (RA4-b/c/d):** practicar los **comandos de movimiento** y el **pick & place con ventosa** usando una clase `BrazoSimulado` (el equivalente en Python al **simulador DobotLab**). Así validamos el código **sin el brazo real**, que en el aula solo lleva montada la **ventosa**.


### Python en un vistazo (lo justo para esta sesión)

| Lo que verás | Qué significa |
|---|---|
| `# texto` | **Comentario**: Python lo ignora; es una nota para ti |
| `nombre = valor` | **Guarda** un valor con un nombre |
| `class Nombre:` | Crea una **clase** (una plantilla de objetos) |
| `def nombre(...):` | Crea una **función** (o un método de la clase) |
| `[a, b, c]` | Una **lista**: valores ordenados |
| `for ... in ...:` | **Repetición** (bucle) |
| `print(...)` | Escribe en pantalla |

Ejecuta las celdas de arriba a abajo; los comentarios `#` explican cada línea.


## 1. El brazo simulado (`BrazoSimulado`)

`BrazoSimulado` imita la API real (`pydobot` / `dType`) y **valida los rangos**: X entre 150 y 320 mm, radio ≤ 320 mm, Z entre −80 y 140 mm. Registra las órdenes y dibuja la trayectoria con matplotlib. Igual que el **simulador DobotLab** (Basic scene), pero en Python.


In [ ]:
%matplotlib inline
import math
import matplotlib.pyplot as plt

class BrazoSimulado:
    """Imita la API de pydobot y valida el script SIN hardware."""
    X_MIN, X_MAX = 150.0, 320.0     # rango de X permitido
    RADIO_MAX = 320.0               # alcance del DOBOT Magician
    Z_MIN, Z_MAX = -80.0, 140.0     # limites de Z

    def __init__(self):
        self.pos = [200.0, 0.0, 50.0, 0.0]
        self.ventosa_activa = False
        self.traza = []

    def _validar(self, x, y, z, r):
        if not (self.X_MIN <= x <= self.X_MAX):
            raise ValueError(f"X={x} fuera de [{self.X_MIN}, {self.X_MAX}]")
        if math.hypot(x, y) > self.RADIO_MAX:
            raise ValueError(f"radio {math.hypot(x, y):.1f} > {self.RADIO_MAX}")
        if not (self.Z_MIN <= z <= self.Z_MAX):
            raise ValueError(f"Z={z} fuera de [{self.Z_MIN}, {self.Z_MAX}]")

    def move_to(self, x, y, z, r=0.0, wait=True):
        """Mueve la punta a (x, y, z) en mm (equivalente a MoveL / MoveJ)."""
        self._validar(x, y, z, r)
        self.pos = [float(x), float(y), float(z), float(r)]
        self.traza.append(("move", float(x), float(y), float(z)))

    def suction(self, activar=True):
        """Activa/desactiva la ventosa (en pydobot: robot.suck(True/False))."""
        self.ventosa_activa = bool(activar)
        self.traza.append(("suction", activar))

    def wait(self, ms):
        self.traza.append(("wait", ms))

    def home(self):
        self.move_to(200, 0, 50, 0)

    def resumen(self):
        return f"ordenes={len(self.traza)} ventosa={'activa' if self.ventosa_activa else 'off'} pos={self.pos}"

    def dibujar_trayectoria(self, titulo="Trayectoria XY", z_corte=None):
        """Dibuja la trayectoria; si z_corte no es None, solo los puntos con z <= z_corte."""
        pts = [(t[1], t[2]) for t in self.traza if t[0] == "move" and (z_corte is None or t[3] <= z_corte)]
        plt.figure(figsize=(5, 5))
        if pts:
            xs, ys = zip(*pts)
            plt.plot(xs, ys, "-o", markersize=3)
        plt.scatter([0], [0], c="k", marker="x", label="base")
        plt.gca().add_patch(plt.Circle((0, 0), self.RADIO_MAX, fill=False, ls="--", color="gray"))
        plt.axis("equal"); plt.grid(True); plt.title(titulo)
        plt.xlabel("X (mm)"); plt.ylabel("Y (mm)"); plt.legend()
        plt.show()


## 2. Comandos de movimiento (MoveJ / MoveL / Jump)

- **MoveJ** — movimiento **articular** (cada eje a su velocidad). Rápido, trayectoria impredecible entre puntos.
- **MoveL** — movimiento **lineal** en cartesianas. Trayectoria recta, más lento.
- **Jump** — sube, se desplaza y baja en un arco (útil para no arrastrar la pieza).

En `BrazoSimulado`, `move_to(x, y, z)` equivale a un movimiento a un punto; encadenar varios puntos con `wait` construye la secuencia.


In [ ]:
brazo = BrazoSimulado()

# Secuencia tipo «Jump»: subir, desplazarse, bajar (sin arrastrar la pieza)
brazo.move_to(250, 0, 60)     # punto seguro (alto)
brazo.move_to(250, 0, 10)     # bajar a la pieza
brazo.wait(500)
brazo.move_to(250, 0, 60)     # subir de nuevo
brazo.move_to(280, 40, 60)    # desplazarse en alto
brazo.move_to(280, 40, 10)    # bajar en destino

print(brazo.resumen())
brazo.dibujar_trayectoria("Secuencia tipo Jump")


## 3. Dibujar (como «Escribir/Dibujar» del simulador)

En el **simulador DobotLab** se dibuja con la función **Escribir/Dibujar**. Aquí lo reproducimos en Python: mover con Z alto, bajar a Z de dibujo y recorrer los vértices. `dibujar_trayectoria(z_corte=...)` muestra solo el trazo (Z baja).


In [ ]:
brazo = BrazoSimulado()
vertices = [(250, -25), (250, 25), (300, 25), (300, -25), (250, -25)]   # cuadrado

brazo.move_to(vertices[0][0], vertices[0][1], 20)   # posicionar arriba (Z=20)
for (x, y) in vertices:                              # recorrer con Z=0 (dibujando)
    brazo.move_to(x, y, 0)

print(brazo.resumen())
brazo.dibujar_trayectoria("Cuadrado (trazo con Z=0)", z_corte=0)


## 4. Ventosa: coger y soltar (pick & place)

En el brazo real solo hay **ventosa** (`suck(True/False)`), que **aspira** superficies lisas y no porosas. La secuencia canónica es: *punto seguro → bajar → aspirar → esperar → subir → desplazar → bajar → soltar → subir*.


In [ ]:
brazo = BrazoSimulado()

# Coger en (240, 0) y dejar en (240, 60)
brazo.move_to(240, 0, 60)     # pre-pick (alto)
brazo.move_to(240, 0, 10)     # pick (bajar a la pieza)
brazo.suction(True)           # aspirar
brazo.wait(500)               # dejar que haga vacío
brazo.move_to(240, 0, 60)     # subir con la pieza
brazo.move_to(240, 60, 60)    # pre-place (alto)
brazo.move_to(240, 60, 10)    # place (bajar)
brazo.suction(False)          # soltar
brazo.move_to(240, 60, 60)    # subir (pieza depositada)

print(brazo.resumen())
brazo.dibujar_trayectoria("Pick & place con ventosa")


## Actividad · Prueba el código y concluye

1. **Prueba el código** cambiando parámetros y volviendo a ejecutar:
   - **Movimiento:** cambia los puntos y alturas de la secuencia tipo *Jump*.
   - **Dibujo:** dibuja otra figura (triángulo, pentágono, iniciales) cambiando la lista `vertices`.
   - **Ventosa:** mueve la pieza a otro destino y comprueba la trayectoria.
2. **Prueba a fallar:** pide un punto con `X < 150` o radio > 320 y mira qué error lanza `BrazoSimulado`.
3. **Añade una conclusión de 3–4 líneas**: por qué conviene validar **sin el robot** (offline) y qué diferencia hay entre MoveJ y MoveL.
